# RS-Solve: Full Benchmark Generation (Kaggle Cloud Worker)
**Đề tài nghiên cứu:** Dự đoán có chọn lọc dựa trên khả năng phân giải cho MLLM ảnh viễn thám (RGSP - IEEE TGRS).

Notebook này tự động quét 5 bộ dữ liệu viễn thám (DOTA, iSAID, DIOR, VisDrone, xView) gắn trong `/kaggle/input/` và sinh tập câu hỏi trắc nghiệm quy mô lớn (**Mục tiêu: 30.000 - 40.000 mẫu**) theo đúng chuẩn thiết kế khoa học của bài báo.

In [ ]:
# 1. Tải mã nguồn mới nhất từ GitHub
import os, sys, shutil, subprocess
from pathlib import Path

repo_dir = Path("/kaggle/working/RGSP-nckh")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

!git clone https://github.com/kttt294/RGSP-nckh.git
!pip install -q Pillow
print("Clone và cài đặt môi trường thành công!")

In [ ]:
# 2. Kiểm tra các bộ dữ liệu đã được mount trong /kaggle/input
input_dir = Path("/kaggle/input")
print("=== DANH SÁCH BỘ DỮ LIỆU ĐÃ GẮN TRONG /kaggle/input ===")
if input_dir.is_dir():
    for sub in sorted(input_dir.iterdir()):
        if sub.is_dir():
            fcount = sum(1 for _ in sub.rglob("*") if _.is_file())
            print(f"  [+] {sub.name}: {fcount} files")

In [ ]:
# 3. Chạy toàn bộ bộ test kiểm thử chất lượng trước khi sinh dữ liệu
os.chdir("/kaggle/working/RGSP-nckh/rs-solve")
!python -m unittest test_parsers.py
!python -m unittest test_label_parser.py

In [ ]:
# 4. Sinh tập dữ liệu Benchmark toàn diện (Target: 30k-40k câu hỏi)
print("=== BẮT ĐẦU QUY TRÌNH SINH TẬP CÂU HỎI QUY MÔ LỚN ===")
cmd = [
    sys.executable, "build_dataset.py",
    "--root-dir", "/kaggle/input",
    "--per-kind", "6000",
    "--output-prefix", "/kaggle/working/rs_solve_full_benchmark",
    "--allow-partial"
]
ret = subprocess.run(cmd)
print(f"Quy trình hoàn tất với mã trạng thái: {ret.returncode}")

In [ ]:
# 5. Thống kê và kiểm tra kết quả xuất ra
import json
report_file = Path("/kaggle/working/rs_solve_full_benchmark_report.json")
jsonl_file = Path("/kaggle/working/rs_solve_full_benchmark.jsonl")

if jsonl_file.exists():
    total_q = sum(1 for _ in jsonl_file.open(encoding="utf-8"))
    size_mb = jsonl_file.stat().st_size / (1024 * 1024)
    print("=== KẾT QUẢ SINH TẬP DỮ LIỆU ===")
    print(f"File benchmark: {jsonl_file.name}")
    print(f"Tổng số câu hỏi sinh được: {total_q} câu")
    print(f"Dung lượng file: {size_mb:.2f} MB")

if report_file.exists():
    report = json.loads(report_file.read_text(encoding="utf-8"))
    print("\nPhân bổ số lượng theo từng loại câu hỏi (Q1 - Q6):")
    for k, v in report.get("selected_counts", {}).items():
        print(f"  - {k}: {v} câu")
    print(f"\nSố lượng ảnh/scene đã xử lý: {len(report.get('image_provenance', {}))}")
    if report.get("rejection_counts"):
        print("\nTop lý do loại bỏ ứng viên (chất lượng kiểm soát):")
        for r, cnt in sorted(report["rejection_counts"].items(), key=lambda x: x[1], reverse=True)[:5]:
            print(f"  - [{cnt} lần]: {r}")